This notebook builds the training data: for each collection, it saves a table listing every usable field with its crop type, its train/validation/test split (taken from FTW's chips), and its AEF and OlmoEarth embeddings.

In [ ]:
import os
import json
import time
import fsspec
import shapely
import numpy as np
import pandas as pd
import pyarrow as pa
import multiprocessing
import geopandas as gpd
from pathlib import Path
import pyarrow.compute as pc
import pyarrow.parquet as pq
from urllib.parse import urljoin
from IPython.display import display
from concurrent.futures import ProcessPoolExecutor
pd.set_option('display.max_rows',20)

In [ ]:
CATALOGURL  = 'https://data.source.coop/ftw/benchmark-data/catalog.json'
RAWDIR      = Path('../data/raw')
INTERIMDIR  = Path('../data/interim')
PROCDIR     = Path('../data/processed')
SETTINGS    = json.loads((INTERIMDIR/'settings.json').read_text())
COLLECTIONS = pd.read_csv(INTERIMDIR/'collections.csv').collection.tolist()
MINFIELDS   = SETTINGS['minfields']
FILECOLUMNS = SETTINGS['filecolumns']
MAXFIELDS   = 200000
MINPIXELS   = 5
NCORES      = len(os.sched_getaffinity(0))
REMOTE      = fsspec.filesystem('https')

pa.set_cpu_count(NCORES)
pa.set_io_thread_count(NCORES)

In [ ]:
def build_file_url(collection,filetype):
    return urljoin(CATALOGURL,f"{collection}/{collection.split('/')[1]}_{filetype}.parquet")

def build_raw_path(collection,filetype):
    return RAWDIR/f"{collection.split('/')[1]}_{filetype}.parquet"

def save_raw_copy(collection,filetype):
    rawpath = build_raw_path(collection,filetype)
    if not rawpath.exists():
        partpath = rawpath.with_suffix('.part')
        with REMOTE.open(build_file_url(collection,filetype)) as file:
            parquet = pq.ParquetFile(file)
            schema  = pa.schema([parquet.schema_arrow.field(col) for col in FILECOLUMNS[filetype]],metadata=parquet.schema_arrow.metadata)
            with pq.ParquetWriter(partpath,schema,compression='zstd') as writer:
                for batch in parquet.iter_batches(batch_size=100000,columns=FILECOLUMNS[filetype]):
                    writer.write_batch(batch)
        partpath.rename(rawpath)

def drop_fields(fields,kept,reason,log):
    log[reason] = len(fields)-len(kept)
    return kept

In [ ]:
def build_table(collection):
    for filetype in FILECOLUMNS:
        save_raw_copy(collection,filetype)
    log = {'collection':collection}
    fields = pq.read_table(build_raw_path(collection,'fields'),columns=['id','hcat:name']).rename_columns(['id','crop'])
    bigids = pq.read_table(build_raw_path(collection,'field_embeddings'),columns=['id'],
                           filters=(pc.field('pixels')>=MINPIXELS)&(pc.field('pixel_rule')!='touched'))['id']
    idcounts = pc.value_counts(fields['id'])
    dupeids  = idcounts.field('values').filter(pc.greater(idcounts.field('counts'),1))
    log['start'] = len(fields)
    fields = drop_fields(fields,fields.filter(pc.is_valid(fields['crop'])),'nolabel',log)
    fields = drop_fields(fields,fields.filter(pc.invert(pc.match_substring_regex(fields['crop'],'^(not_known_and_other|other_arable_land_crops|unspecified_.*)$'))),'catchall',log)
    fields = drop_fields(fields,fields.filter(pc.invert(pc.is_in(fields['id'],value_set=dupeids))),'duplicateid',log)
    fields = drop_fields(fields,fields.filter(pc.is_in(fields['id'],value_set=bigids)),'small',log)
    crops  = fields['crop'].dictionary_encode().to_pandas()
    fields = drop_fields(fields,fields.filter((crops.map(crops.value_counts()).astype(int)>=MINFIELDS).to_numpy()),'rarecrop',log)
    crops  = fields['crop'].dictionary_encode().to_pandas()
    quota  = (crops.value_counts()*min(1,MAXFIELDS/len(crops))).round().clip(lower=MINFIELDS)
    rank   = crops.sample(frac=1,random_state=100).groupby(crops,observed=True).cumcount().sort_index()
    fields = drop_fields(fields,fields.filter((rank<crops.map(quota).astype(int)).to_numpy()),'capped',log).to_pandas()
    ids    = fields.id.to_numpy()

    stats     = ['aef_mean','aef_std','olmoearth_mean','olmoearth_std']
    names     = [f'{model}{i:03d}' for model in ('aef','olmoearth') for i in range(128)]
    vectors   = pq.read_table(build_raw_path(collection,'field_embeddings'),columns=['id']+stats,filters=pc.field('id').isin(ids))
    features  = pd.DataFrame(np.hstack([pc.list_flatten(vectors[stat]).to_numpy().reshape(vectors.num_rows,-1) for stat in stats]),columns=names)
    features.insert(0,'id',vectors['id'].to_numpy(zero_copy_only=False))
    polygons  = gpd.read_parquet(build_raw_path(collection,'fields'),columns=['id','geometry'],filters=pc.field('id').isin(ids))
    centroids = gpd.GeoDataFrame({'id':polygons.id},geometry=shapely.centroid(np.asarray(polygons.geometry)),crs=polygons.crs)
    chips     = gpd.read_parquet(build_raw_path(collection,'chips')).rename(columns={'id':'chip'})
    location  = gpd.sjoin(centroids,chips,predicate='within',how='left').drop_duplicates('id')
    location  = pd.DataFrame({'id':location.id,'chip':location.chip,'split':location.split,'lon':location.geometry.x,'lat':location.geometry.y})

    table = fields.merge(location,on='id').merge(features,on='id')
    table = drop_fields(table,table[table.split.notna()],'outsidechips',log)
    table.to_parquet(PROCDIR/f"fields_{collection.split('/')[1]}.parquet",index=False)
    log.update(kept=len(table),crops=table.crop.nunique(),**table.split.value_counts().reindex(['train','val','test']).to_dict())
    return log

In [ ]:
todo    = [collection for collection in COLLECTIONS if not (PROCDIR/f"fields_{collection.split('/')[1]}.parquet").exists()]
logpath = INTERIMDIR/'build_log.csv'
print(f'Building {len(todo)} of {len(COLLECTIONS)} collections...')

for collection in todo:
    start = time.perf_counter()
    with ProcessPoolExecutor(1,mp_context=multiprocessing.get_context('fork')) as executor:
        log = executor.submit(build_table,collection).result()
    pd.DataFrame([log]).to_csv(logpath,mode='a',header=not logpath.exists(),index=False)
    minutes,seconds = divmod(round(time.perf_counter()-start),60)
    print(f'- Saved {PROCDIR}/fields_{collection.split("/")[1]}.parquet ({minutes}m {seconds}s)')

In [ ]:
log     = pd.read_csv(INTERIMDIR/'build_log.csv').drop_duplicates('collection',keep='last')
headers = {'collection':'Collection',
           'start':'Fields in Source File',
           'nolabel':'No Crop Label*',
           'catchall':'Catch-All Class*',
           'duplicateid':'Duplicate Field ID*',
           'small':f'< {MINPIXELS} Pixels*',
           'rarecrop':f'Crop with < {MINFIELDS} Fields*',
           'capped':f'> {MAXFIELDS:,}-Field Cap*',
           'outsidechips':'Outside Every Chip*',
           'kept':'Fields Kept',
           'crops':'Crop Classes Kept',
           'train':'Training Fields',
           'val':'Validation Fields',
           'test':'Test Fields'}

display(log[list(headers)].rename(columns=headers).style.format('{:,.0f}',subset=list(headers.values())[1:]).hide(axis='index'))